In [1]:
print("hello")

hello


In [19]:
import pandas as pd
import numpy as np

raw = pd.read_csv("U_of_DE_Comp.csv")
df = raw.copy()          # raw stays untouched
print(df.shape)

# Loading and making a working copy 

(300000, 46)


In [20]:
no_file = df["Acct1"] == 99
print("No-file rows:", no_file.sum())
print("Their default rate:", df.loc[no_file, "bad_flg_90plus_24m"].mean())  # about 20%

df = df[~no_file].copy()

No-file rows: 1037
Their default rate: 0.20154291224686596


## Finding: applicants with no credit file
- 1,037 applicants have no credit file (placeholder codes in nearly every column)
- Their default rate is about 20.2%, versus about 6.3% for everyone else
- Decision: removed from the cleaned data; saved separately in removed_no_credit_file.csv

In [22]:
no_file = raw["Acct1"] == 99

print(f"No-file rows:         {no_file.sum()}")
print(f"No-file default rate: {raw.loc[no_file, 'bad_flg_90plus_24m'].mean():.4f}")
print(f"Everyone else:        {raw.loc[~no_file, 'bad_flg_90plus_24m'].mean():.4f}")
print(f"Overall:              {raw['bad_flg_90plus_24m'].mean():.4f}")

No-file rows:         1037
No-file default rate: 0.2015
Everyone else:        0.0626
Overall:              0.0631


# Shows the Comparison between the No file rows default rate versus eevryone else

In [23]:
sentinels = {
    "Acct11": [9998, 9996, 9999],
    "Acct13": [9998, 9996, 9999],
    "Del6":   [9996, 9999],
    "Acct5":  [9999],
    "CO3":    [999999999],
    "Score2": [9003],
}
for col, codes in sentinels.items():
    df[col] = df[col].replace(codes, np.nan)

# turn placeholder codes into blanks

In [24]:
df = df.drop(columns=["Acct14", "Acct8", "PubRec1"])   # ID, and two all-zero columns
df["Acct12"] = df["Acct12"].astype("category")

# drop useless columns

In [25]:
print(df.shape)
print(df.isna().sum()[df.isna().sum() > 0])   # columns that now have blanks
df.describe().T[["min", "max"]]                # look for leftover 999s or 9999s

(298963, 43)
Acct11    294483
Acct13    268494
Del6      204132
Score2      3474
dtype: int64


,min,max
Acct1,0.0,84.0
Acct2,0.0,98.0
Acct3,0.0,90.0
Acct4,0.0,31.0
Acct5,0.0,9994.0
Acct6,0.0,84.0
Acct7,0.0,994.0
Acct9,0.0,18.0
Acct10,0.0,84.0
Acct11,0.0,276.0


In [26]:
more_codes = {
    "Acct2": [98], "Acct5": [9994], "Acct7": [994], "CO3": [999999996],
    "Del2": [97], "Del3": [97], "Del4": [97], "Del5": [97], "Del9": [997],
    "Score2": [9002], "Util1": [996], "Util2": [997],
    "Util3": [996, 997, 998], "Util4": [98],
}

for col, codes in more_codes.items():
    df[col + "_nodata"] = df[col].isin(codes).astype(int)   # flag first
    df[col] = df[col].replace(codes, np.nan)                # then blank it

# the earlier blanks in Del6 mean "never delinquent", so flag them too
df["Del6_never"] = df["Del6"].isna().astype(int)

# Fixing additional variables

In [27]:
print(df.shape)

# columns with blanks, and how many
blanks = df.isna().sum()
print(blanks[blanks > 0])

# min and max of every column
df.describe().T[["min", "max"]]

(298963, 58)
Acct2      23432
Acct5          1
Acct7       1946
Acct11    294483
Acct13    268494
CO3         2241
Del2        3566
Del3        3566
Del4        1718
Del5        2123
Del6      204132
Del9        2123
Score2      7146
Util1       9713
Util2       4090
Util3      42668
Util4      23432
dtype: int64


,min,max
Acct1,0.0,84.0
Acct2,0.0,65.0
Acct3,0.0,90.0
Acct4,0.0,31.0
Acct5,0.0,839.0
Acct6,0.0,84.0
Acct7,0.0,100.0
Acct9,0.0,18.0
Acct10,0.0,84.0
Acct11,0.0,276.0


# fixing additonal cells that have inconclusive or not probable data

In [28]:
extra = {"Util1": [995], "Util2": [995, 996], "Util3": [995]}

for col, codes in extra.items():
    df[col + "_nodata"] = ((df[col + "_nodata"] == 1) | df[col].isin(codes)).astype(int)
    df[col] = df[col].replace(codes, np.nan)

In [29]:
print(df[["Util1", "Util2", "Util3"]].max())
print(df[["Util1", "Util2", "Util3"]].isna().sum())

Util1    990.0
Util2    990.0
Util3    362.0
dtype: float64
Util1     9981
Util2    10641
Util3    42682
dtype: int64


In [30]:
for col in ["Util1", "Util2"]:
    df[col] = df[col].replace(990, np.nan)

# continuing to fix strange data points

In [32]:
import os
os.makedirs("BHDAC-Fall-26-Case-Comp", exist_ok=True)
df.to_csv("BHDAC-Fall-26-Case-Comp/clean.csv", index=False)

# Saving a clean version of the dataset and putting it in the correct folder